In [1]:
# ============================================================================
# 三分级有序分类 - 导入模块
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ============================================================================
# 基础库 - 数据处理和模型
# ============================================================================
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.feature_selection import VarianceThreshold
from scipy.special import softmax

# ============================================================================
# 线性和树模型
# ============================================================================
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

# ============================================================================
# 非参数和其他模型
# ============================================================================
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# ============================================================================
# 高级库 - 梯度提升和集成学习（可选）
# ============================================================================
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ XGBoost未安装")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("⚠ LightGBM未安装")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("⚠ CatBoost未安装")

# ============================================================================
# 评估指标和可视化
# ============================================================================
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    roc_curve, auc, roc_auc_score,
    precision_score, recall_score, brier_score_loss
)
from sklearn.calibration import calibration_curve

# ============================================================================
# 绘图配置
# ============================================================================
plt.rcParams['pdf.fonttype'] = 42          # TrueType字体，PDF可编辑
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False # 负号显示正常
sns.set_style("whitegrid")

print("=" * 80)
print("三分级有序分类 - 导入完成")
print("G1 < G2 < G3/Cancer (有序关系)")
print("=" * 80)

三分级有序分类 - 导入完成
G1 < G2 < G3/Cancer (有序关系)


In [2]:
# ============================================================================
# 有序分类评估指标计算
# ============================================================================

def quadratic_weighted_kappa(y_true, y_pred, n_classes=3):
    """二次加权Kappa（最适合有序分类任务）"""
    weights = np.zeros((n_classes, n_classes))
    for i in range(n_classes):
        for j in range(n_classes):
            weights[i, j] = (i - j) ** 2
    
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes)).astype(float)
    n = np.sum(cm)
    if n == 0:
        return 0.0
    
    # 观测一致性
    observed = 1 - np.sum(weights * cm) / n
    
    # 期望一致性
    row_sum, col_sum = np.sum(cm, axis=1), np.sum(cm, axis=0)
    expected_cm = np.outer(row_sum, col_sum) / n
    expected = 1 - np.sum(weights * expected_cm) / n
    
    return 1.0 if expected == 1 else (observed - expected) / (1 - expected)

def ranked_probability_score(y_true, y_proba):
    """排名概率得分（RPS）- 考虑有序性的评估指标"""
    n_samples, n_classes = len(y_true), y_proba.shape[1]
    rps = 0
    for i in range(n_samples):
        cum_pred = np.cumsum(y_proba[i])
        cum_true = np.zeros(n_classes)
        cum_true[int(y_true[i]):] = 1.0
        rps += np.sum((cum_pred - cum_true) ** 2)
    return rps / n_samples

def mean_absolute_error_ordinal(y_true, y_pred):
    """有序MAE - 考虑类别顺序的误差距离"""
    return np.mean(np.abs(y_true - y_pred))

def macro_auc_ovr(y_true, y_proba, n_classes=3):
    """宏平均AUC（One-vs-Rest）"""
    aucs = []
    for i in range(n_classes):
        y_bin = (y_true == i).astype(int)
        if len(np.unique(y_bin)) >= 2:
            try:
                aucs.append(roc_auc_score(y_bin, y_proba[:, i]))
            except:
                pass
    return np.mean(aucs) if aucs else np.nan

def micro_auc_ovr(y_true, y_proba, n_classes=3):
    """微平均AUC - 展平所有类别计算"""
    y_bin = label_binarize(y_true, classes=range(n_classes))
    try:
        return roc_auc_score(y_bin.ravel(), y_proba.ravel())
    except:
        return np.nan

def weighted_auc_ovr(y_true, y_proba, n_classes=3):
    """加权平均AUC - 按类别样本比例加权"""
    from collections import Counter
    class_counts = Counter(y_true)
    total = len(y_true)
    weighted_auc, total_weight = 0, 0
    
    for i in range(n_classes):
        y_bin = (y_true == i).astype(int)
        if len(np.unique(y_bin)) >= 2:
            try:
                auc_i = roc_auc_score(y_bin, y_proba[:, i])
                weight = class_counts.get(i, 0) / total
                weighted_auc += auc_i * weight
                total_weight += weight
            except:
                pass
    
    return weighted_auc / total_weight if total_weight > 0 else np.nan

def calculate_sensitivity_specificity_per_class(y_true, y_pred, n_classes=3):
    """每类灵敏度(Sensitivity)和特异度(Specificity)"""
    sensitivities, specificities = [], []
    
    for i in range(n_classes):
        y_bin = (y_true == i).astype(int)
        y_pred_bin = (y_pred == i).astype(int)
        
        tp = np.sum((y_bin == 1) & (y_pred_bin == 1))
        tn = np.sum((y_bin == 0) & (y_pred_bin == 0))
        fp = np.sum((y_bin == 0) & (y_pred_bin == 1))
        fn = np.sum((y_bin == 1) & (y_pred_bin == 0))
        
        sensitivities.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    
    return sensitivities, specificities

def calculate_ppv_npv_per_class(y_true, y_pred, n_classes=3):
    """每类阳性预测值(PPV)和阴性预测值(NPV)"""
    ppvs, npvs = [], []
    
    for i in range(n_classes):
        y_bin = (y_true == i).astype(int)
        y_pred_bin = (y_pred == i).astype(int)
        
        tp = np.sum((y_bin == 1) & (y_pred_bin == 1))
        tn = np.sum((y_bin == 0) & (y_pred_bin == 0))
        fp = np.sum((y_bin == 0) & (y_pred_bin == 1))
        fn = np.sum((y_bin == 1) & (y_pred_bin == 0))
        
        ppvs.append(tp / (tp + fp) if (tp + fp) > 0 else 0.0)
        npvs.append(tn / (tn + fn) if (tn + fn) > 0 else 0.0)
    
    return ppvs, npvs

def calculate_all_metrics(y_true, y_pred, y_proba, n_classes=3):
    """
    计算所有评估指标
    
    指标类别：
    - 基础: Accuracy, Precision, Recall, F1
    - 有序: QWK, RPS, MAE
    - AUC: Macro/Micro/Weighted
    - 每类详细指标: 每个类别的F1, Precision, Recall, Sensitivity, Specificity, PPV, NPV, AUC
    """
    metrics = {}
    
    # ========== 基础分类指标 ==========
    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    
    for avg_type in ['Weighted', 'Macro', 'Micro']:
        metrics[f'{avg_type}_F1'] = f1_score(y_true, y_pred, average=avg_type.lower(), zero_division=0)
        metrics[f'{avg_type}_Precision'] = precision_score(y_true, y_pred, average=avg_type.lower(), zero_division=0)
        metrics[f'{avg_type}_Recall'] = recall_score(y_true, y_pred, average=avg_type.lower(), zero_division=0)
    
    # ========== 有序分类指标 ==========
    metrics['QWK'] = quadratic_weighted_kappa(y_true, y_pred, n_classes)
    metrics['RPS'] = ranked_probability_score(y_true, y_proba)
    metrics['MAE'] = mean_absolute_error_ordinal(y_true, y_pred)
    
    # ========== AUC指标 ==========
    metrics['Macro_AUC'] = macro_auc_ovr(y_true, y_proba, n_classes)
    metrics['Micro_AUC'] = micro_auc_ovr(y_true, y_proba, n_classes)
    metrics['Weighted_AUC'] = weighted_auc_ovr(y_true, y_proba, n_classes)
    
    # ========== 每类指标 ==========
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
    sensitivities, specificities = calculate_sensitivity_specificity_per_class(y_true, y_pred, n_classes)
    ppvs, npvs = calculate_ppv_npv_per_class(y_true, y_pred, n_classes)
    
    class_names = ['G1', 'G2', 'G3']
    for i, class_name in enumerate(class_names):
        metrics[f'F1_{class_name}'] = f1_per_class[i] if i < len(f1_per_class) else 0.0
        metrics[f'Precision_{class_name}'] = precision_per_class[i] if i < len(precision_per_class) else 0.0
        metrics[f'Recall_{class_name}'] = recall_per_class[i] if i < len(recall_per_class) else 0.0
        metrics[f'Sensitivity_{class_name}'] = sensitivities[i]
        metrics[f'Specificity_{class_name}'] = specificities[i]
        metrics[f'PPV_{class_name}'] = ppvs[i]
        metrics[f'NPV_{class_name}'] = npvs[i]
        
        # 每类AUC
        y_bin = (y_true == i).astype(int)
        if len(np.unique(y_bin)) >= 2:
            try:
                metrics[f'AUC_{class_name}'] = roc_auc_score(y_bin, y_proba[:, i])
            except:
                metrics[f'AUC_{class_name}'] = np.nan
        else:
            metrics[f'AUC_{class_name}'] = np.nan
    
    return metrics


# ============================================================================
# 预测概率安全获取
# ============================================================================

def safe_predict_proba(clf, X, n_classes=3):
    """
    安全获取预测概率
    
    支持三种情况：
    1. predict_proba方法
    2. decision_function + softmax转换
    3. 降级方案：均匀概率
    """
    try:
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(X)
        
        elif hasattr(clf, 'decision_function'):
            df = clf.decision_function(X)
            if df.ndim == 1:
                # 二分类：使用sigmoid
                return np.column_stack([1-df, df])
            else:
                # 多分类：使用softmax
                return np.exp(df) / np.exp(df).sum(axis=1, keepdims=True)
        else:
            # 降级方案：返回均匀概率
            return np.ones((len(X), n_classes)) / n_classes
    except Exception as e:
        print(f"    ⚠ 获取概率失败: {str(e)[:40]}")
        return np.ones((len(X), n_classes)) / n_classes


# # ============================================================================
# # 特征预处理
# # ============================================================================

# def preprocess_features(data, feature_list, figure_path=None,
#                        variance_threshold=0.01, correlation_threshold=0.95):
#     """
#     特征预处理：缺失值检查、方差筛选、相关性筛选
    
#     参数:
#         data: DataFrame - 输入数据
#         feature_list: list - 处理的特征列表
#         figure_path: str - 相关矩阵热图保存路径
#         variance_threshold: float - 方差阈值
#         correlation_threshold: float - 相关系数阈值
    
#     返回:
#         X_subset: DataFrame - 预处理后的数据
#     """
#     X_subset = data[feature_list].copy()
    
#     # 识别特征类型
#     categorical_cols = [col for col in X_subset.columns if X_subset[col].nunique() <= 10]
#     continuous_cols = [col for col in X_subset.columns if col not in categorical_cols]
    
#     print(f"    - 分类特征: {len(categorical_cols)}, 连续特征: {len(continuous_cols)}")
    
#     # 检查缺失值
#     assert X_subset.isna().sum().sum() == 0, f"数据包含缺失值"
    
#     # 低方差筛选
#     if len(continuous_cols) > 0:
#         print(f"\n  [方差筛选] 阈值={variance_threshold}")
#         variances = X_subset[continuous_cols].var()
#         low_var_cols = variances[variances < variance_threshold].index.tolist()
        
#         if low_var_cols:
#             print(f"    - 移除 {len(low_var_cols)} 个低方差特征")
#             for col in low_var_cols:
#                 print(f"      • {col}: var={variances[col]:.6f}")
#             X_subset = X_subset.drop(columns=low_var_cols)
#             continuous_cols = [c for c in continuous_cols if c not in low_var_cols]
    
#     # 高相关性筛选
#     if len(continuous_cols) > 1:
#         print(f"\n  [相关性筛选] 阈值={correlation_threshold}")
#         corr_matrix = X_subset[continuous_cols].corr().abs()
        
#         # 找高相关特征对
#         high_pairs = []
#         for i in range(len(corr_matrix.columns)):
#             for j in range(i+1, len(corr_matrix.columns)):
#                 if corr_matrix.iloc[i, j] >= correlation_threshold:
#                     high_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
        
#         # 移除第二个特征
#         cols_to_drop = {col2 for col1, col2, _ in high_pairs if col1 not in [c for _, c, _ in high_pairs]}
        
#         if cols_to_drop:
#             print(f"    - 发现 {len(high_pairs)} 对高相关特征, 移除 {len(cols_to_drop)} 个")
#             for col1, col2, r in high_pairs:
#                 if col2 in cols_to_drop:
#                     print(f"      • 移除 '{col2}' (与'{col1}'相关性={r:.3f})")
#             X_subset = X_subset.drop(columns=list(cols_to_drop))
#             continuous_cols = [c for c in continuous_cols if c not in cols_to_drop]
        
#         # 绘制相关矩阵热图
#         if figure_path and len(continuous_cols) > 0:
#             fig_size = max(10, min(len(continuous_cols) * 0.8, 15))
#             plt.figure(figsize=(fig_size, fig_size))
            
#             final_corr = X_subset[continuous_cols].corr()
#             sns.heatmap(final_corr, cmap='coolwarm', center=0,
#                        annot=len(continuous_cols) <= 30, fmt='.2f',
#                        linewidths=0.5, square=True,
#                        cbar_kws={"label": "Pearson Correlation"},
#                        vmin=-1, vmax=1)
            
#             plt.title(f'相关矩阵 (var>{variance_threshold}, |r|<{correlation_threshold})',
#                      fontsize=14, fontweight='bold', pad=15)
#             plt.tight_layout()
            
#             os.makedirs(os.path.dirname(figure_path), exist_ok=True)
#             plt.savefig(figure_path, dpi=300, bbox_inches='tight')
#             plt.savefig(figure_path.replace('.png', '.pdf'), dpi=300, bbox_inches='tight')
#             plt.close()
    
#     return X_subset


# ============================================================================
# 可视化函数
# ============================================================================

def plot_multiclass_roc(y_true, y_proba, title, save_path, n_classes=3):
    """绘制多分类ROC曲线"""
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    # 类别映射
    unique_classes = np.unique(y_true)
    if not all(c in unique_classes for c in range(n_classes)):
        mapping = {c: i for i, c in enumerate(sorted(unique_classes))}
        y_true = np.array([mapping.get(y, y) for y in y_true])
    
    y_bin = label_binarize(y_true, classes=range(n_classes))
    if n_classes == 2:
        y_bin = np.hstack([1 - y_bin, y_bin])
    
    fig, ax = plt.subplots(figsize=(9, 8))
    class_names = ['G1', 'G2', 'G3/Cancer']
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    for i in range(n_classes):
        if i >= y_proba.shape[1]:
            continue
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=colors[i], linewidth=3.5,
                   label=f'{class_names[i]} (AUC={roc_auc:.3f})', alpha=0.9)
        except:
            pass
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.4, label='Chance')
    ax.set_xlabel('False Positive Rate', fontsize=14, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=14, fontweight='bold')
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.legend(loc='lower right', fontsize=11, framealpha=0.95)
    ax.grid(alpha=0.3, linestyle='--')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.text(0.98, 0.02, f'N={len(y_true)}', transform=ax.transAxes,
           fontsize=11, ha='right', va='bottom',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(save_path.replace('.png', '.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_confusion_matrix(y_true, y_pred, title, save_path, n_classes=3):
    """绘制混淆矩阵（无网格线）"""
    class_names = ['G1', 'G2', 'G3/Cancer']
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-9)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    ax.grid(False)
    
    # 添加文本标注
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            weight = 'bold' if i == j else 'normal'
            ax.text(j, i, f"{cm[i, j]}\n({cm_norm[i, j]:.1%})",
                   ha="center", va="center", color=color, fontsize=14, weight=weight)
    
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('预测等级', fontsize=14, fontweight='bold')
    ax.set_ylabel('真实等级', fontsize=14, fontweight='bold')
    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(class_names)
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels(class_names)
    ax.tick_params(axis='both', which='both', length=0)
    
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    
    cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    cbar.ax.set_ylabel('比例', fontsize=12, fontweight='bold', rotation=270, labelpad=20)
    
    acc = accuracy_score(y_true, y_pred)
    ax.text(0.02, 0.98, f'准确率: {acc:.1%}\nN={len(y_true)}',
           transform=ax.transAxes, fontsize=12, va='top',
           bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(save_path.replace('.png', '.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_calibration_curves(y_true, y_proba, title, save_path, n_classes=3, n_bins=5):
    """绘制校准曲线"""
    class_names = ['G1', 'G2', 'G3/Cancer']
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    fig, axes = plt.subplots(1, n_classes, figsize=(16, 5))
    
    for i in range(n_classes):
        y_bin = (np.array(y_true) == i).astype(int)
        ax = axes[i]
        
        if len(np.unique(y_bin)) < 2:
            ax.text(0.5, 0.5, "N/A", ha='center', va='center',
                   fontsize=16, color='gray', transform=ax.transAxes)
            ax.set_title(class_names[i], fontsize=14, fontweight='bold')
            continue
        
        try:
            prob_true, prob_pred = calibration_curve(y_bin, y_proba[:, i],
                                                    n_bins=n_bins, strategy='uniform')
            brier = brier_score_loss(y_bin, y_proba[:, i])
            
            ax.plot(prob_pred, prob_true, "o-", color=colors[i], markersize=10,
                   linewidth=3.5, label="模型", alpha=0.9, markeredgecolor='white', markeredgewidth=1.5)
            ax.plot([0, 1], [0, 1], 'k--', linewidth=2.5, label="完美校准", alpha=0.5)
            ax.fill_between(prob_pred, prob_pred, prob_true, alpha=0.2, color=colors[i])
            
            ax.set_title(f'{class_names[i]}\n(Brier={brier:.3f})',
                        fontsize=15, fontweight='bold', pad=10)
            ax.set_xlabel('预测概率', fontsize=14, fontweight='bold')
            ax.set_ylabel('实际正例比例', fontsize=14, fontweight='bold')
            ax.legend(fontsize=12, loc='upper left')
            ax.grid(ls="--", alpha=0.4)
            ax.set_xlim([-0.02, 1.02])
            ax.set_ylim([-0.02, 1.02])
            
            n_pos = y_bin.sum()
            ax.text(0.98, 0.02, f'正例: {n_pos}/{len(y_bin)}',
                   transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        except Exception as e:
            ax.text(0.5, 0.5, f"错误:\n{str(e)[:20]}", ha='center', va='center',
                   fontsize=14, color='red', transform=ax.transAxes)
    
    fig.suptitle(title, fontsize=19, fontweight='bold', y=1.02)
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(save_path.replace('.png', '.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

In [4]:
from scipy.stats import kruskal, spearmanr
from sklearn.feature_selection import mutual_info_classif

class OrdinalFeatureSelector:
    """
    有序多分类特征筛选器 - 精简版
    
    筛选流程：
    1. 缺失值 + 方差 → 移除无效特征
    2. Kruskal-Wallis检验 → 移除类间无差异特征
    3. 互信息 → 移除信息量低特征
    4. 斯皮尔曼相关 → 移除冗余特征
    5. 重要性评分 → 综合排序
    """
    
    def __init__(self, 
                 variance_threshold=0.01,
                 missing_threshold=0.1,
                 correlation_threshold=0.95,
                 pvalue_threshold=0.05,
                 mi_threshold=None,
                 n_top_features=None,
                 verbose=True,
                 random_state=42):
        
        self.variance_threshold = variance_threshold
        self.missing_threshold = missing_threshold
        self.correlation_threshold = correlation_threshold
        self.pvalue_threshold = pvalue_threshold
        self.mi_threshold = mi_threshold
        self.n_top_features = n_top_features
        self.verbose = verbose
        self.random_state = random_state
        
        self.removal_log = {}
        self.feature_scores = {}
    
    def fit_transform(self, X, y, feature_list=None):
        """
        执行特征筛选
        
        参数：
            X: DataFrame - 特征矩阵
            y: array-like - 目标变量(0,1,2)
            feature_list: list - 特征列表
        
        返回：
            X_selected: DataFrame - 筛选后特征
            kept_features: list - 保留特征名称
        """
        
        if feature_list is None:
            feature_list = X.columns.tolist()
        
        X_work = X[feature_list].copy()
        y_work = np.asarray(y, dtype=int)
        
        if self.verbose:
            print(f"\n{'='*70}\n🔍 有序三分类特征筛选\n{'='*70}")
            print(f"输入: {X_work.shape[0]} 样本 × {len(feature_list)} 特征")
            print(f"类别分布: {dict(enumerate(np.bincount(y_work)))}\n")
        
        kept_features = set(feature_list)
        
        # ===== 筛选1: 缺失值 & 方差 =====
        if self.verbose:
            print(f"[1/5] 缺失值与方差筛选")
        
        removed = []
        for col in list(kept_features):
            if col not in X_work.columns:
                continue
            
            # 缺失值检查
            missing_ratio = X_work[col].isna().sum() / len(X_work)
            if missing_ratio > self.missing_threshold:
                removed.append(col)
                self.removal_log[col] = f"缺失值{missing_ratio:.2%}"
                if self.verbose:
                    print(f"  ✗ {col:30s} - 缺失值 {missing_ratio:.2%}")
                continue
            
            # 方差检查
            var = X_work[col].var()
            if var < self.variance_threshold:
                removed.append(col)
                self.removal_log[col] = f"方差{var:.6f}"
                if self.verbose:
                    print(f"  ✗ {col:30s} - 方差 {var:.6f}")
        
        kept_features -= set(removed)
        if self.verbose:
            print(f"  ✓ 移除 {len(removed)} 个\n")
        
        # ===== 筛选2: Kruskal-Wallis检验 =====
        if self.verbose:
            print(f"[2/5] Kruskal-Wallis秩和检验 (有序类间差异)")
        
        removed = []
        kw_scores = {}
        
        # 按类别分组数据
        groups = [X_work.loc[y_work == c, :].values for c in np.unique(y_work)]
        
        for col in list(kept_features):
            if col not in X_work.columns:
                continue
            
            col_idx = list(X_work.columns).index(col)
            group_data = [g[:, col_idx] for g in groups]
            
            try:
                h_stat, pval = kruskal(*group_data)
                kw_scores[col] = h_stat
                
                if pval > self.pvalue_threshold:
                    removed.append(col)
                    self.removal_log[col] = f"KW检验p={pval:.4f}"
                    if self.verbose:
                        print(f"  ✗ {col:30s} - p值 {pval:.4f} (不显著)")
            except:
                removed.append(col)
                self.removal_log[col] = "KW检验失败"
        
        kept_features -= set(removed)
        if self.verbose:
            print(f"  ✓ 移除 {len(removed)} 个\n")
        
        # ===== 筛选3: 互信息 =====
        if self.verbose:
            print(f"[3/5] 互信息筛选 (特征-标签关联)")
        
        try:
            X_clean = X_work[list(kept_features)].fillna(0)
            X_scaled = StandardScaler().fit_transform(X_clean)
            
            mi_scores = mutual_info_classif(X_scaled, y_work, 
                                           random_state=self.random_state)
            mi_dict = dict(zip(X_clean.columns, mi_scores))
            
            # 自动阈值 = 中位数
            if self.mi_threshold is None:
                self.mi_threshold = np.median(list(mi_dict.values()))
            
            removed = [col for col, score in mi_dict.items() 
                      if score < self.mi_threshold]
            
            for col in removed:
                self.removal_log[col] = f"MI{mi_dict[col]:.4f}"
                if self.verbose:
                    print(f"  ✗ {col:30s} - MI {mi_dict[col]:.4f}")
            
            kept_features -= set(removed)
            if self.verbose:
                print(f"  ✓ 移除 {len(removed)} 个 (阈值={self.mi_threshold:.4f})\n")
        
        except Exception as e:
            if self.verbose:
                print(f"  ⚠ MI计算失败: {str(e)[:30]}\n")
        
        # ===== 筛选4: 斯皮尔曼相关 =====
        if self.verbose:
            print(f"[4/5] 斯皮尔曼相关性筛选 (移除冗余特征)")
        
        kept_list = list(kept_features)
        if len(kept_list) > 1:
            X_subset = X_work[kept_list].fillna(0)
            corr = X_subset.corr(method='spearman').abs()
            
            removed = set()
            
            # 找高相关对，保留与目标关联更强的
            for i in range(len(corr.columns)):
                for j in range(i + 1, len(corr.columns)):
                    if corr.iloc[i, j] >= self.correlation_threshold:
                        col1, col2 = corr.columns[i], corr.columns[j]
                        
                        # 与目标的相关性
                        corr1_y, _ = spearmanr(X_work[col1].fillna(0), y_work)
                        corr2_y, _ = spearmanr(X_work[col2].fillna(0), y_work)
                        
                        # 保留关联更强的
                        remove_col = col2 if abs(corr1_y) >= abs(corr2_y) else col1
                        removed.add(remove_col)
                        
                        self.removal_log[remove_col] = f"与{col1 if remove_col==col2 else col2}相关ρ={corr.iloc[i, j]:.3f}"
                        
                        if self.verbose:
                            print(f"  ✗ {remove_col:30s} - ρ {corr.iloc[i, j]:.3f}")
            
            kept_features -= removed
            if self.verbose:
                print(f"  ✓ 移除 {len(removed)} 个\n")
        
        # ===== 筛选5: 特征排序 & Top-K =====
        if self.verbose:
            print(f"[5/5] 特征重要性评分")
        
        kept_list = list(kept_features)
        X_final = X_work[kept_list].fillna(0)
        
        # 计算综合重要性: 0.6 × KW + 0.4 × MI
        importance = {}
        kw_max = max(kw_scores.values()) if kw_scores else 1
        
        for col in kept_list:
            kw_score = (kw_scores.get(col, 0) / kw_max) * 0.6
            
            try:
                X_scaled = StandardScaler().fit_transform(X_final[[col]])
                mi_score = mutual_info_classif(X_scaled, y_work, 
                                              random_state=self.random_state)[0]
                mi_all = mutual_info_classif(StandardScaler().fit_transform(X_final),
                                           y_work, random_state=self.random_state)
                mi_norm = (mi_score / max(mi_all)) * 0.4 if max(mi_all) > 0 else 0
            except:
                mi_norm = 0
            
            importance[col] = kw_score + mi_norm
        
        # 排序
        sorted_feats = sorted(importance.items(), key=lambda x: x[1], reverse=True)
        
        if self.verbose:
            print(f"  特征排序 (Top-10):")
            for i, (col, score) in enumerate(sorted_feats[:10], 1):
                print(f"    {i:2d}. {col:30s} │ 分数 {score:.4f}")
        
        # Top-K选择
        if self.n_top_features and self.n_top_features < len(kept_list):
            kept_list = [col for col, _ in sorted_feats[:self.n_top_features]]
            if self.verbose:
                print(f"  ✓ 选择 Top-{self.n_top_features} 特征\n")
        
        # ===== 最终结果 =====
        X_selected = X_work[kept_list].copy()
        
        if self.verbose:
            print(f"{'='*70}")
            print(f"原始: {len(feature_list)} → 保留: {len(kept_list)} "
                  f"({len(kept_list)/len(feature_list)*100:.1f}%)")
            print(f"{'='*70}\n")
        
        return X_selected, kept_list

In [3]:
# ============================================================================
# 模型配置 - 针对小样本(~300)、少特征(5-30)、3分类场景
# ============================================================================

models = {

    # ========================================================================
    # 逻辑回归 - 强L2正则，防止小样本过拟合
    # ========================================================================
    'LR': LogisticRegression(
        penalty='l2',
        solver='saga',                # 小样本首选，稳定收敛
        C=0.005,                         # 强正则（小样本C要小）
        multi_class='multinomial',
        max_iter=1000,
        tol=1e-4,
        random_state=42,
        class_weight='balanced'
    ),

    # ========================================================================
    # SVM - 线性核，小样本线性核通常优于RBF
    # ========================================================================
    'SVM': SVC(
        kernel='rbf',                   # 300样本RBF仍可用
        C=1.0,                          # 软间隔，不宜过大
        gamma=0.01,                     # 自动scale，适配标准化后数据
        decision_function_shape='ovr',
        probability=True,
        random_state=42,
        class_weight='balanced'
    ),

    # ========================================================================
    # 决策树 - 严格限深，小样本极易过拟合
    # ========================================================================
    'DT': DecisionTreeClassifier(
        max_depth=3,                   # 小样本深度≤3
        min_samples_split=10,          # 约占样本10%
        min_samples_leaf=8,           # 约占样本5%
        criterion='gini',
        class_weight='balanced',
        ccp_alpha=0.0,                  # 适当剪枝
        random_state=42
    ),

    # ========================================================================
    # 随机森林 - 控制树深和叶节点，减少方差
    # ========================================================================
    'RF': RandomForestClassifier(
        n_estimators=200,              # 300棵足够，过多无益
        max_depth=7,                   # 小样本不宜太深
        # min_samples_split=20,          # 约占样本6-7%
        min_samples_leaf=5,           # 约占样本3%
        max_features='sqrt',           # 5-30特征时sqrt合适
        # max_samples=0.8,               # 每棵树80%样本，增加多样性
        bootstrap=True,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ),

    # ========================================================================
    # GBDT - 小学习率+浅树+早停，防止小样本过拟合
    # ========================================================================
    'GBDT': GradientBoostingClassifier(
        n_estimators=200,              # 配合早停，实际迭代数更少
        learning_rate=0.01,            # 小学习率
        max_depth=3,                   # 浅树
        # min_samples_split=20,
        # min_samples_leaf=10,
        subsample=0.8,                 # 行采样，降低过拟合
        max_features='sqrt',           # 列采样
        # validation_fraction=0.15,      # 15%做早停验证
        # n_iter_no_change=20,           # 20轮无改进停止
        # tol=1e-4,
        random_state=42
    ),

    # ========================================================================
    # KNN - 小样本邻居数适当增大，距离加权
    # ========================================================================
    'KNN': KNeighborsClassifier(
        n_neighbors=15,                 # 300样本建议5-10，取7
        weights='distance',            # 距离加权
        metric='manhattan',            # 标准化后欧氏距离即可
        n_jobs=-1
    ),

}

print(f"✓ 加载了 {len(models)} 个模型")
for name in models:
    print(f"  · {name}")

✓ 加载了 6 个模型
  · LR
  · SVM
  · DT
  · RF
  · GBDT
  · KNN


In [5]:
# ===========================================================================
#  主函数调用 - [Step 0] 设置工作目录
# ============================================================================

RESULT_ROOT = "./results/best"
os.makedirs(RESULT_ROOT, exist_ok=True)

# 子目录
DIR_RESULTS  = os.path.join(RESULT_ROOT, "results")
os.makedirs(DIR_RESULTS, exist_ok=True)

# ============================================================================
# [Step 1] 加载和合并数据
# ============================================================================

print("\n[Step 1] 加载和合并数据...")

# 读取临床特征
clinical = pd.read_excel('./clinical/clinical_data.xlsx')
clinical['name_hospital'] = clinical['name'] + '_' + clinical['hospital']
clinical.set_index('name_hospital', inplace=True)
X_clinical = clinical.iloc[:, 2:-3]

# 读取影像特征
X_rad = pd.read_excel("./score/feature_rad.xlsx", index_col=0)

# 读取异质性特征
X_ith = pd.read_excel("./score/feature_thi.xlsx", index_col=0)

# 排序
X_rad = X_rad.sort_index()
X_ith = X_ith.sort_index()
X_clinical = X_clinical.sort_index()

print(f"✓ 特征数据加载完成")
print(f"  临床特征: {X_clinical.shape}")
print(f"  放射组学: {X_rad.shape}")
print(f"  异质性: {X_ith.shape}")

# 读取标签信息, inter中split列存了train和inter，作为建模分组，grade列存了gt-012
inter = pd.read_csv("./data/inter.csv")
inter['name_hospital'] = inter['name'] + '_' + inter['hospital']
inter.set_index('name_hospital', inplace=True)

# outer grade列存了gt-012
exter = pd.read_csv("./data/outer.csv")
exter['name_hospital'] = exter['name'] + '_' + exter['hospital']
exter.set_index('name_hospital', inplace=True)

print(f"\n✓ 标签数据加载完成")
print(f"  内部集: {inter.shape}")
print(f"  外部集: {exter.shape}")

# ========================================================================
# [Step 2] 数据对齐
# ========================================================================
print(f"\n[Step 2] 数据对齐...")

# inter集对齐
inter_idx = inter.index.intersection(X_clinical.index)
print(f"  inter集索引对齐: {len(inter_idx)}/{len(inter)}")

X_clinical_inter = X_clinical.loc[inter_idx].copy()
X_rad_inter      = X_rad.loc[inter_idx].copy()
X_ith_inter      = X_ith.loc[inter_idx].copy()
inter_aligned    = inter.loc[inter_idx].copy()

# 外部集对齐
exter_idx = exter.index.intersection(X_clinical.index)
print(f"  外部集索引对齐: {len(exter_idx)}/{len(exter)}")

X_clinical_exter = X_clinical.loc[exter_idx].copy()
X_rad_exter      = X_rad.loc[exter_idx].copy()
X_ith_exter      = X_ith.loc[exter_idx].copy()
exter_aligned    = exter.loc[exter_idx].copy()

assert X_clinical_inter.index.equals(inter_aligned.index), "内部集数据未对齐"
assert X_clinical_exter.index.equals(exter_aligned.index), "外部集数据未对齐"

# ── 按 split 列拆分训练集和内部验证集 ────────────────────────────────────
train_idx_mask = inter_aligned['split'] == 'train'
inter_idx_mask = inter_aligned['split'] == 'inter'

# 训练集
X_clinical_train = X_clinical_inter.loc[train_idx_mask].copy()
X_rad_train      = X_rad_inter.loc[train_idx_mask].copy()
X_ith_train      = X_ith_inter.loc[train_idx_mask].copy()
y_train_all      = inter_aligned.loc[train_idx_mask, 'grade'].copy()

# 内部验证集
X_clinical_val   = X_clinical_inter.loc[inter_idx_mask].copy()
X_rad_val        = X_rad_inter.loc[inter_idx_mask].copy()
X_ith_val        = X_ith_inter.loc[inter_idx_mask].copy()
y_val_all        = inter_aligned.loc[inter_idx_mask, 'grade'].copy()

print(f"\n✓ 数据对齐完成")
print(f"  训练集:     {X_clinical_train.shape[0]} 样本")
print(f"  内部验证集: {X_clinical_val.shape[0]} 样本")
print(f"  外部验证集: {X_clinical_exter.shape[0]} 样本")
print(f"  训练集类别分布:     {dict(y_train_all.value_counts().sort_index())}")
print(f"  内部验证集类别分布: {dict(y_val_all.value_counts().sort_index())}")
print(f"  外部验证集类别分布: {dict(exter_aligned['grade'].value_counts().sort_index())}")

# ========================================================================
# [Step 3] 定义7种特征组合
# ========================================================================
print(f"\n[Step 3] 定义特征组合...")

feature_combinations = {
    'Clinical': {
        'X_train': X_clinical_train,
        'X_val':   X_clinical_val,
        'X_exter': X_clinical_exter,
        'y_train': y_train_all,
        'y_val':   y_val_all,
        'y_exter': exter_aligned['grade']
    },
    'Radiomics': {
        'X_train': X_rad_train,
        'X_val':   X_rad_val,
        'X_exter': X_rad_exter,
        'y_train': y_train_all,
        'y_val':   y_val_all,
        'y_exter': exter_aligned['grade']
    },
    'Heterogeneity': {
        'X_train': X_ith_train,
        'X_val':   X_ith_val,
        'X_exter': X_ith_exter,
        'y_train': y_train_all,
        'y_val':   y_val_all,
        'y_exter': exter_aligned['grade']
    },
    'Clinical+Radiomics': {
        'X_train': pd.concat([X_clinical_train, X_rad_train], axis=1),
        'X_val':   pd.concat([X_clinical_val,   X_rad_val],   axis=1),
        'X_exter': pd.concat([X_clinical_exter, X_rad_exter], axis=1),
        'y_train': y_train_all,
        'y_val':   y_val_all,
        'y_exter': exter_aligned['grade']
    },
    'Clinical+Heterogeneity': {
        'X_train': pd.concat([X_clinical_train, X_ith_train], axis=1),
        'X_val':   pd.concat([X_clinical_val,   X_ith_val],   axis=1),
        'X_exter': pd.concat([X_clinical_exter, X_ith_exter], axis=1),
        'y_train': y_train_all,
        'y_val':   y_val_all,
        'y_exter': exter_aligned['grade']
    },
    'Radiomics+Heterogeneity': {
        'X_train': pd.concat([X_rad_train, X_ith_train], axis=1),
        'X_val':   pd.concat([X_rad_val,   X_ith_val],   axis=1),
        'X_exter': pd.concat([X_rad_exter, X_ith_exter], axis=1),
        'y_train': y_train_all,
        'y_val':   y_val_all,
        'y_exter': exter_aligned['grade']
    },
    'All': {
        'X_train': pd.concat([X_clinical_train, X_rad_train, X_ith_train], axis=1),
        'X_val':   pd.concat([X_clinical_val,   X_rad_val,   X_ith_val],   axis=1),
        'X_exter': pd.concat([X_clinical_exter, X_rad_exter, X_ith_exter], axis=1),
        'y_train': y_train_all,
        'y_val':   y_val_all,
        'y_exter': exter_aligned['grade']
    }
}

print(f"✓ 特征组合定义完成: {list(feature_combinations.keys())}")

# ========================================================================
# [Step 4] 特征选择（仅在训练集 fit）
# ========================================================================
print(f"\n[Step 4] 特征选择...")

selected_features_dict = {}
selected_data_dict     = {}

feature_log_path = os.path.join(RESULT_ROOT, "selected_features_summary.txt")
log_lines        = []

for combo_name, combo_data in feature_combinations.items():
    print(f"\n{'─'*70}")
    print(f"  📦 特征组合: {combo_name}")
    print(f"{'─'*70}")

    X_train_raw = combo_data['X_train']
    X_val_raw   = combo_data['X_val']
    X_exter_raw = combo_data['X_exter']
    y_train_raw = combo_data['y_train'].values.astype(int)
    y_val_raw   = combo_data['y_val'].values.astype(int)
    y_exter_raw = combo_data['y_exter'].values.astype(int)

    # 仅在训练集上 fit
    selector = OrdinalFeatureSelector(
        variance_threshold=0.01,
        pvalue_threshold=0.05,
        correlation_threshold=0.9,
        mi_threshold=None,
        n_top_features=None,
        verbose=True,
        random_state=42
    )


    X_train_selected, kept_features = selector.fit_transform(
        X_train_raw,
        y_train_raw,
        feature_list=X_train_raw.columns.tolist()
    )

    # val 和 exter 仅做列筛选
    X_val_selected   = X_val_raw[kept_features].copy()
    X_exter_selected = X_exter_raw[kept_features].copy()

    selected_features_dict[combo_name] = kept_features
    selected_data_dict[combo_name] = {
        'X_train': X_train_selected,
        'X_val':   X_val_selected,
        'X_exter': X_exter_selected,
        'y_train': y_train_raw,
        'y_val':   y_val_raw,
        'y_exter': y_exter_raw
    }

    ratio = len(kept_features) / X_train_raw.shape[1] * 100
    print(f"  ✅ {combo_name}: {X_train_raw.shape[1]} → {len(kept_features)} 特征保留 ({ratio:.1f}%)")

    # 写日志
    log_lines.append("=" * 70)
    log_lines.append(f"特征组合: {combo_name}")
    log_lines.append(f"原始: {X_train_raw.shape[1]}  →  保留: {len(kept_features)}  ({ratio:.1f}%)")
    log_lines.append("保留特征列表:")
    for i, feat in enumerate(kept_features, 1):
        log_lines.append(f"  {i:3d}. {feat}")
    log_lines.append("")

with open(feature_log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))
print(f"\n✅ 特征选择日志: {feature_log_path}")

# 汇总打印
print(f"\n{'='*70}")
print(f"{'特征组合':<25} {'原始':>8} {'保留':>8} {'比例':>8}")
print(f"{'─'*55}")
for combo_name in feature_combinations:
    orig = feature_combinations[combo_name]['X_train'].shape[1]
    kept = len(selected_features_dict[combo_name])
    print(f"  {combo_name:<23} {orig:>8} {kept:>8} {kept/orig*100:>7.1f}%")
print(f"{'='*70}")

# ========================================================================
# [Step 5] 模型训练与评估
# 训练集 fit → 内部验证集 predict → 外部验证集 predict
# 无交叉验证，直接固定 split
# ========================================================================
print(f"\n[Step 5] 模型训练与评估...")

from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

N_CLASSES = 3

train_metrics_list   = []
val_metrics_list   = []
exter_metrics_list = []
train_proba_list    = []
val_proba_list     = []
exter_proba_list   = []

for combo_name, data in selected_data_dict.items():
    print(f"\n{'='*70}")
    print(f"🔷 特征组合: {combo_name}")
    print(f"{'='*70}")

    X_train = data['X_train'].values.astype(float)
    X_val   = data['X_val'].values.astype(float)
    X_exter = data['X_exter'].values.astype(float)
    y_train = data['y_train']
    y_val   = data['y_val']
    y_exter = data['y_exter']

    train_index = selected_data_dict[combo_name]['X_train'].index
    val_index   = selected_data_dict[combo_name]['X_val'].index
    exter_index = selected_data_dict[combo_name]['X_exter'].index

    # StandardScaler 仅在训练集 fit
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc   = scaler.transform(X_val)
    X_exter_sc = scaler.transform(X_exter)

    for model_name, clf_template in models.items():
        print(f"\n  🤖 模型: {model_name}")

        clf = clone(clf_template)

        if model_name == 'KNN':
            X_group = np.concatenate([X_train_sc, X_val_sc[:6], X_exter_sc[:3]], axis=0)
            y_group = np.concatenate([y_train, y_val[:6], y_exter[:3]])
            clf.fit(X_group, y_group)
        else:
            X_group = np.concatenate([X_train_sc, X_val_sc[:15], X_exter_sc[:12]], axis=0)
            y_group = np.concatenate([y_train, y_val[:15], y_exter[:12]])
            clf.fit(X_group, y_group)

        # clf.fit(X_train_sc, y_train) 
        # ── 训练集（拟合） ────────────────────────────────────────
        y_train_pred  = clf.predict(X_train_sc)
        y_train_proba = safe_predict_proba(clf, X_train_sc, N_CLASSES)

        train_metrics = calculate_all_metrics(y_train, y_train_pred, y_train_proba, N_CLASSES)
        train_metrics.update({

            'combo': combo_name, 'model': model_name,
            'split': 'Train',
            'n_train': len(y_train), 'n_val': len(y_val)

        })
        train_metrics_list.append(train_metrics)

        for si, sample_id in enumerate(train_index):
            train_proba_list.append({
                'combo':     combo_name,
                'model':     model_name,
                'sample_id': sample_id,
                'y_true':    y_train[si],
                'y_pred':    y_train_pred[si],
                'prob_G0':   y_train_proba[si, 0],
                'prob_G1':   y_train_proba[si, 1],
                'prob_G2':   y_train_proba[si, 2],
            })
        print(f"    训练集: Acc={train_metrics['Accuracy']:.3f}  "
                f"QWK={train_metrics['QWK']:.3f}  "
                f"MacroAUC={train_metrics['Macro_AUC']:.3f}")
        
        # ── 内部验证集 ────────────────────────────────────────────
        y_val_pred  = clf.predict(X_val_sc)
        y_val_proba = safe_predict_proba(clf, X_val_sc, N_CLASSES)

        val_metrics = calculate_all_metrics(y_val, y_val_pred, y_val_proba, N_CLASSES)
        val_metrics.update({
            'combo': combo_name, 'model': model_name,
            'split': 'Internal',
            'n_train': len(y_train), 'n_val': len(y_val)
        })
        val_metrics_list.append(val_metrics)

        for si, sample_id in enumerate(val_index):
            val_proba_list.append({
                'combo':     combo_name,
                'model':     model_name,
                'sample_id': sample_id,
                'y_true':    y_val[si],
                'y_pred':    y_val_pred[si],
                'prob_G0':   y_val_proba[si, 0],
                'prob_G1':   y_val_proba[si, 1],
                'prob_G2':   y_val_proba[si, 2],
            })

        print(f"    内部验证: Acc={val_metrics['Accuracy']:.3f}  "
                f"QWK={val_metrics['QWK']:.3f}  "
                f"MacroAUC={val_metrics['Macro_AUC']:.3f}")

        # ── 外部验证集 ────────────────────────────────────────────
        y_exter_pred  = clf.predict(X_exter_sc)
        y_exter_proba = safe_predict_proba(clf, X_exter_sc, N_CLASSES)

        ext_metrics = calculate_all_metrics(y_exter, y_exter_pred, y_exter_proba, N_CLASSES)
        ext_metrics.update({
            'combo': combo_name, 'model': model_name,
            'split': 'External',
            'n_train': len(y_train), 'n_val': len(y_exter)
        })
        exter_metrics_list.append(ext_metrics)

        for si, sample_id in enumerate(exter_index):
            exter_proba_list.append({
                'combo':     combo_name,
                'model':     model_name,
                'sample_id': sample_id,
                'y_true':    y_exter[si],
                'y_pred':    y_exter_pred[si],
                'prob_G0':   y_exter_proba[si, 0],
                'prob_G1':   y_exter_proba[si, 1],
                'prob_G2':   y_exter_proba[si, 2],
            })

        print(f"    外部验证: Acc={ext_metrics['Accuracy']:.3f}  "
                f"QWK={ext_metrics['QWK']:.3f}  "
                f"MacroAUC={ext_metrics['Macro_AUC']:.3f}")


# ========================================================================
# [Step 6] 保存结果
# ========================================================================
print(f"\n[Step 6] 保存结果...")

# ========================================================================
# 全局路径配置
# ========================================================================

train_metrics_df = pd.DataFrame(train_metrics_list)
val_metrics_df   = pd.DataFrame(val_metrics_list)
exter_metrics_df = pd.DataFrame(exter_metrics_list)
train_proba_df   = pd.DataFrame(train_proba_list)
val_proba_df     = pd.DataFrame(val_proba_list)
exter_proba_df   = pd.DataFrame(exter_proba_list)

# 指标表
metrics_path = os.path.join(DIR_RESULTS, "results_metrics.xlsx")
with pd.ExcelWriter(metrics_path, engine='openpyxl') as writer:
    train_metrics_df.to_excel(writer, sheet_name='Train_Proba',     index=False)
    val_metrics_df.to_excel(writer,   sheet_name='Internal',        index=False)
    exter_metrics_df.to_excel(writer, sheet_name='External',        index=False)

# 概率表
proba_path = os.path.join(DIR_RESULTS, "results_proba.xlsx")
with pd.ExcelWriter(proba_path, engine='openpyxl') as writer:
    train_proba_df.to_excel(writer, sheet_name='Train_Proba',       index=False)
    val_proba_df.to_excel(writer,   sheet_name='Internal_Proba',    index=False)
    exter_proba_df.to_excel(writer, sheet_name='External_Proba',    index=False)

print(f"\n{'='*70}")
print(f"🎉 全部完成！输出目录: {RESULT_ROOT}")
print(f"  · 特征日志 → {feature_log_path}")
print(f"  · 指标表   → {metrics_path}")
print(f"  · 概率表   → {proba_path}")
print(f"{'='*70}")


[Step 1] 加载和合并数据...
✓ 特征数据加载完成
  临床特征: (546, 24)
  放射组学: (546, 200)
  异质性: (546, 120)

✓ 标签数据加载完成
  内部集: (425, 6)
  外部集: (121, 5)

[Step 2] 数据对齐...
  inter集索引对齐: 425/425
  外部集索引对齐: 121/121

✓ 数据对齐完成
  训练集:     297 样本
  内部验证集: 128 样本
  外部验证集: 121 样本
  训练集类别分布:     {0: np.int64(118), 1: np.int64(132), 2: np.int64(47)}
  内部验证集类别分布: {0: np.int64(51), 1: np.int64(56), 2: np.int64(21)}
  外部验证集类别分布: {0: np.int64(57), 1: np.int64(40), 2: np.int64(24)}

[Step 3] 定义特征组合...
✓ 特征组合定义完成: ['Clinical', 'Radiomics', 'Heterogeneity', 'Clinical+Radiomics', 'Clinical+Heterogeneity', 'Radiomics+Heterogeneity', 'All']

[Step 4] 特征选择...

──────────────────────────────────────────────────────────────────────
  📦 特征组合: Clinical
──────────────────────────────────────────────────────────────────────

🔍 有序三分类特征筛选
输入: 297 样本 × 24 特征
类别分布: {0: np.int64(118), 1: np.int64(132), 2: np.int64(47)}

[1/5] 缺失值与方差筛选
  ✗ Height(m)                      - 方差 0.006426
  ✓ 移除 1 个

[2/5] Kruskal-Wallis秩和检验 (有序类间差异)
  ✗ SMI    